# Camada Prata: Limpeza e Normalização  

A camada Prata aplica transformações para tornar os dados confiáveis e consistentes:
* Tipagem correta
* Remoção de duplicatas
* Normalização de texto
* Tratamento de nulos

# Configuração do Ambiente

In [0]:
########################################################
#### LIBS
########################################################

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# Listagem dos arquivos nos volumes  

Lista dos arquivos no volume especificado. Esses dados serão tratamos e carregados na camada silver.

In [0]:
%sql
LIST "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/"

path,name,size,modification_time
/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/alunos.parquet,alunos.parquet,71447882,1787021320000
/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/meta_alfabetizacao_brasil.parquet,meta_alfabetizacao_brasil.parquet,9779,1787021317000
/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/meta_alfabetizacao_municipio.parquet,meta_alfabetizacao_municipio.parquet,223852,1787021317000
/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/meta_alfabetizacao_uf.parquet,meta_alfabetizacao_uf.parquet,12755,1787021317000
/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/municipio.parquet,municipio.parquet,499916,1787021317000
/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/uf.parquet,uf.parquet,18226,1787021316000


In [0]:
volume_batch = "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/"

# Extract file names into a Python list
dados_batch = [file.name for file in dbutils.fs.ls(volume_batch)]
dados_batch

['alunos.parquet',
 'meta_alfabetizacao_brasil.parquet',
 'meta_alfabetizacao_municipio.parquet',
 'meta_alfabetizacao_uf.parquet',
 'municipio.parquet',
 'uf.parquet']

# Leitura dos dados

In [0]:
########################################################
#### LEITURA DOS DADOS EM STREAMING E BATCH
########################################################

###
def leitura_streaming_bronze(dados:str):
    return spark.read.format("delta").load(f"/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/streaming/{dados}")

###alunos = spark.read.parquet("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/alunos.parquet")
def leitura_batch_bronze(dados:str):
    return spark.read.parquet(f"/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/{dados}")

# Funções

## Remoção dos metadados


In [0]:
########################################################
#### REMOÇÃO DOS METADADOS - INICIAM COM "_"
########################################################
def remover_coluna_metadados_bronze(dados):
    remover_metadados = [col for col in dados.columns if not col.startswith("_")]

    print("Colunas antes:", len(dados.columns))
    print("Colunas depois:", len(dados.select(remover_metadados).columns))
    
    return dados.select(remover_metadados)


## Remoção de linhas duplicadas

In [0]:
########################################################
#### REMOÇÃO DE LINHAS DUPLICADAS
########################################################
def remover_duplicados(dados):
    dados_tratados =  dados.dropDuplicates()
    print("Linhas antes:", dados.count())
    print("Linhas depois:", dados_tratados.count())
    
    return dados_tratados

## Metadados camada prata

In [0]:
def metadados_silver(dados):
    dados_tratados = (dados
    .withColumn('_prata_data_processamento', F.current_timestamp())
    .withColumn('_versao_pipeline', F.lit('v1.0_spark'))
    )

    print("Colunas antes:", len(dados.columns))
    print("Colunas depois:", len(dados_tratados.columns))

    return dados_tratados

# ETL camada prata

## Alunos

In [0]:
#
alunos_silver = leitura_batch_bronze("alunos.parquet")
#
alunos_silver = remover_coluna_metadados_bronze(alunos_silver)
#
alunos_silver = remover_duplicados(alunos_silver)
#
alunos_silver = metadados_silver(alunos_silver)
#
alunos_silver = (alunos_silver
    .withColumn("ano",                          F.col("ano").cast(IntegerType()))
    .withColumn("id_municipio",                 F.col("id_municipio").cast(StringType()))
    .withColumn("id_escola",                    F.col("id_escola").cast(StringType()))
    .withColumn("id_aluno",                     F.col("id_aluno").cast(StringType()))
    .withColumn("caderno",                      F.col("caderno").cast(IntegerType()))
    .withColumn("serie",                        F.col("serie").cast(IntegerType()))
    .withColumn("presenca",                     F.col("presenca").cast(IntegerType()))
    .withColumn("preenchimento_caderno",        F.col("preenchimento_caderno").cast(IntegerType()))
    .withColumn("alfabetizado",                 F.col("alfabetizado").cast(IntegerType()))
    .withColumn("proficiencia",                 F.col("proficiencia").cast(DoubleType()))
    .withColumn("peso_aluno",                   F.col("peso_aluno").cast(DoubleType()))
    .withColumn("_prata_data_processamento",    F.col("_prata_data_processamento").cast(TimestampType()))
    .withColumn("_versao_pipeline",             F.col("_versao_pipeline").cast(StringType()))
)
#
alunos_silver.show(10)

Colunas antes: 15
Colunas depois: 12
Linhas antes: 3867999
Linhas depois: 3867999
Colunas antes: 12
Colunas depois: 14
+----+------------+---------+--------+-------+-----+----+--------+---------------------+------------+------------+----------+-------------------------+----------------+
| ano|id_municipio|id_escola|id_aluno|caderno|serie|rede|presenca|preenchimento_caderno|alfabetizado|proficiencia|peso_aluno|_prata_data_processamento|_versao_pipeline|
+----+------------+---------+--------+-------+-----+----+--------+---------------------+------------+------------+----------+-------------------------+----------------+
|2024|     3138401| 60019339|31177410|      1|    2|   2|       0|                    0|           0|        NULL|      NULL|     2026-09-21 20:33:...|      v1.0_spark|
|2024|     1302603| 60001246|13050318|      1|    2|   3|       0|                    0|           0|        NULL|      NULL|     2026-09-21 20:33:...|      v1.0_spark|
|2023|     1600535| 60003692|1600928

In [0]:
print(alunos_silver.schema)

StructType([StructField('ano', IntegerType(), True), StructField('id_municipio', StringType(), True), StructField('id_escola', StringType(), True), StructField('id_aluno', StringType(), True), StructField('caderno', IntegerType(), True), StructField('serie', IntegerType(), True), StructField('rede', StringType(), True), StructField('presenca', IntegerType(), True), StructField('preenchimento_caderno', IntegerType(), True), StructField('alfabetizado', IntegerType(), True), StructField('proficiencia', DoubleType(), True), StructField('peso_aluno', DoubleType(), True), StructField('_prata_data_processamento', TimestampType(), False), StructField('_versao_pipeline', StringType(), False)])


In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/alunos"
#
alunos_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)

## Meta Alfabetização Brasil

In [0]:
#
meta_alfabetizacao_brasil_silver = leitura_batch_bronze("meta_alfabetizacao_brasil.parquet")
#
meta_alfabetizacao_brasil_silver = remover_coluna_metadados_bronze(meta_alfabetizacao_brasil_silver)
#
meta_alfabetizacao_brasil_silver = remover_duplicados(meta_alfabetizacao_brasil_silver)
#
meta_alfabetizacao_brasil_silver = metadados_silver(meta_alfabetizacao_brasil_silver)
#
meta_alfabetizacao_brasil_silver = (meta_alfabetizacao_brasil_silver
    .withColumn("ano",                          F.col("ano").cast(IntegerType()))
    .withColumn("rede",                         F.col("rede").cast(StringType()))
    .withColumn("taxa_alfabetizacao",           F.col("taxa_alfabetizacao").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2024",      F.col("meta_alfabetizacao_2024").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2025",      F.col("meta_alfabetizacao_2025").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2026",      F.col("meta_alfabetizacao_2026").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2027",      F.col("meta_alfabetizacao_2027").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2028",      F.col("meta_alfabetizacao_2028").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2029",      F.col("meta_alfabetizacao_2029").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2030",      F.col("meta_alfabetizacao_2030").cast(DoubleType()))
    .withColumn("percentual_participacao",      F.col("percentual_participacao").cast(DoubleType()))
    .withColumn("_prata_data_processamento",    F.col("_prata_data_processamento").cast(TimestampType()))
    .withColumn("_versao_pipeline",             F.col("_versao_pipeline").cast(StringType()))
)
#
meta_alfabetizacao_brasil_silver.show(10)

Colunas antes: 14
Colunas depois: 11
Linhas antes: 3
Linhas depois: 3
Colunas antes: 11
Colunas depois: 13
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------------+----------------+
| ano|   rede|taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|_prata_data_processamento|_versao_pipeline|
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------------+----------------+
|2025|Pública|              66.0|                   60.0|                   64.0|     

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/meta_alfabetizacao_brasil"
#
meta_alfabetizacao_brasil_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)

## Meta Alfabetização Município

In [0]:
#
meta_alfabetizacao_municipio_silver = leitura_batch_bronze("meta_alfabetizacao_municipio.parquet")
#
meta_alfabetizacao_municipio_silver = remover_coluna_metadados_bronze(meta_alfabetizacao_municipio_silver)
#
meta_alfabetizacao_municipio_silver = remover_duplicados(meta_alfabetizacao_municipio_silver)
#
meta_alfabetizacao_municipio_silver = metadados_silver(meta_alfabetizacao_municipio_silver)
#
meta_alfabetizacao_municipio_silver = (meta_alfabetizacao_municipio_silver
    .withColumn("ano",                          F.col("ano").cast(IntegerType()))
    .withColumn("rede",                         F.col("rede").cast(StringType()))
    .withColumn("taxa_alfabetizacao",           F.col("taxa_alfabetizacao").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2024",      F.col("meta_alfabetizacao_2024").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2025",      F.col("meta_alfabetizacao_2025").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2026",      F.col("meta_alfabetizacao_2026").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2027",      F.col("meta_alfabetizacao_2027").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2028",      F.col("meta_alfabetizacao_2028").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2029",      F.col("meta_alfabetizacao_2029").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2030",      F.col("meta_alfabetizacao_2030").cast(DoubleType()))
    .withColumn("nivel_alfabetizacao",          F.col("nivel_alfabetizacao").cast(IntegerType()))
    .withColumn("percentual_participacao",      F.col("percentual_participacao").cast(DoubleType()))
    .withColumn("_prata_data_processamento",    F.col("_prata_data_processamento").cast(TimestampType()))
    .withColumn("_versao_pipeline",             F.col("_versao_pipeline").cast(StringType()))
)
#
meta_alfabetizacao_municipio_silver.show(10)

Colunas antes: 16
Colunas depois: 13
Linhas antes: 10704
Linhas depois: 10704
Colunas antes: 13
Colunas depois: 15
+----+------------+---------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------+-----------------------+-------------------------+----------------+
| ano|id_municipio|     rede|taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|nivel_alfabetizacao|percentual_participacao|_prata_data_processamento|_versao_pipeline|
+----+------------+---------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------+-----------------------+-----------------

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/meta_alfabetizacao_municipio"
#
meta_alfabetizacao_municipio_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)

## Meta Alfabetização UF

In [0]:
#
meta_alfabetizacao_uf_silver = leitura_batch_bronze("meta_alfabetizacao_uf.parquet")
#
meta_alfabetizacao_uf_silver = remover_coluna_metadados_bronze(meta_alfabetizacao_uf_silver)
#
meta_alfabetizacao_uf_silver = remover_duplicados(meta_alfabetizacao_uf_silver)
#
meta_alfabetizacao_uf_silver = metadados_silver(meta_alfabetizacao_uf_silver)
#
meta_alfabetizacao_uf_silver = (meta_alfabetizacao_uf_silver
    .withColumn("ano",                          F.col("ano").cast(IntegerType()))
    .withColumn("rede",                         F.col("rede").cast(StringType()))
    .withColumn("taxa_alfabetizacao",           F.col("taxa_alfabetizacao").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2024",      F.col("meta_alfabetizacao_2024").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2025",      F.col("meta_alfabetizacao_2025").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2026",      F.col("meta_alfabetizacao_2026").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2027",      F.col("meta_alfabetizacao_2027").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2028",      F.col("meta_alfabetizacao_2028").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2029",      F.col("meta_alfabetizacao_2029").cast(DoubleType()))
    .withColumn("meta_alfabetizacao_2030",      F.col("meta_alfabetizacao_2030").cast(DoubleType()))
    .withColumn("percentual_participacao",      F.col("percentual_participacao").cast(DoubleType()))
    .withColumn("_prata_data_processamento",    F.col("_prata_data_processamento").cast(TimestampType()))
    .withColumn("_versao_pipeline",             F.col("_versao_pipeline").cast(StringType()))
)
#
meta_alfabetizacao_uf_silver.show(10)

Colunas antes: 15
Colunas depois: 12
Linhas antes: 81
Linhas depois: 81
Colunas antes: 12
Colunas depois: 14
+----+--------+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------------+----------------+
| ano|sigla_uf|   rede|taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|_prata_data_processamento|_versao_pipeline|
+----+--------+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------------+----------------+
|2024|      AP|Pública|             46.62|               

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/meta_alfabetizacao_uf"
#
meta_alfabetizacao_uf_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)

## Município

In [0]:
#
municipio_silver = leitura_batch_bronze("municipio.parquet")
#
municipio_silver.show(10)

+----+------------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------+--------------+--------------------+
| ano|id_municipio|serie|rede|taxa_alfabetizacao|media_portugues|proporcao_aluno_nivel_0|proporcao_aluno_nivel_1|proporcao_aluno_nivel_2|proporcao_aluno_nivel_3|proporcao_aluno_nivel_4|proporcao_aluno_nivel_5|proporcao_aluno_nivel_6|proporcao_aluno_nivel_7|proporcao_aluno_nivel_8|_momento_ingestao|_data_ingestao|             _origem|
+----+------------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------+--------------+--------------

In [0]:
#
municipio_silver = leitura_batch_bronze("municipio.parquet")
#
municipio_silver = remover_coluna_metadados_bronze(municipio_silver)
#
municipio_silver = remover_duplicados(municipio_silver)
#
municipio_silver = metadados_silver(municipio_silver)
#
municipio_silver = (municipio_silver
    .withColumn("ano",                          F.col("ano").cast(IntegerType()))
    .withColumn("id_municipio",                 F.col("id_municipio").cast(StringType()))
    .withColumn("serie",                        F.col("serie").cast(IntegerType()))
    .withColumn("rede",                         F.col("rede").cast(IntegerType()))
    .withColumn("taxa_alfabetizacao",           F.col("taxa_alfabetizacao").cast(DoubleType()))
    .withColumn("media_portugues",              F.col("media_portugues").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_0",      F.col("proporcao_aluno_nivel_0").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_1",      F.col("proporcao_aluno_nivel_1").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_2",      F.col("proporcao_aluno_nivel_2").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_3",      F.col("proporcao_aluno_nivel_3").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_4",      F.col("proporcao_aluno_nivel_4").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_5",      F.col("proporcao_aluno_nivel_5").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_6",      F.col("proporcao_aluno_nivel_6").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_7",      F.col("proporcao_aluno_nivel_7").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_8",      F.col("proporcao_aluno_nivel_8").cast(DoubleType()))
    .withColumn("_prata_data_processamento",    F.col("_prata_data_processamento").cast(TimestampType()))
    .withColumn("_versao_pipeline",             F.col("_versao_pipeline").cast(StringType()))
)
#
municipio_silver.show(10)

Colunas antes: 18
Colunas depois: 15
Linhas antes: 23995
Linhas depois: 23995
Colunas antes: 15
Colunas depois: 17
+----+------------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------------+----------------+
| ano|id_municipio|serie|rede|taxa_alfabetizacao|media_portugues|proporcao_aluno_nivel_0|proporcao_aluno_nivel_1|proporcao_aluno_nivel_2|proporcao_aluno_nivel_3|proporcao_aluno_nivel_4|proporcao_aluno_nivel_5|proporcao_aluno_nivel_6|proporcao_aluno_nivel_7|proporcao_aluno_nivel_8|_prata_data_processamento|_versao_pipeline|
+----+------------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+--

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/municipio"
#
municipio_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)

## UF

In [0]:
#
uf_silver = leitura_batch_bronze("uf.parquet")
#
uf_silver = remover_coluna_metadados_bronze(uf_silver)
#
uf_silver = remover_duplicados(uf_silver)
#
uf_silver = metadados_silver(uf_silver)
#
uf_silver = (uf_silver
    .withColumn("ano",                          F.col("ano").cast(IntegerType()))
    .withColumn("sigla_uf",                     F.col("sigla_uf").cast(StringType()))
    .withColumn("serie",                        F.col("serie").cast(IntegerType()))
    .withColumn("rede",                         F.col("rede").cast(IntegerType()))
    .withColumn("taxa_alfabetizacao",           F.col("taxa_alfabetizacao").cast(DoubleType()))
    .withColumn("media_portugues",              F.col("media_portugues").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_0",      F.col("proporcao_aluno_nivel_0").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_1",      F.col("proporcao_aluno_nivel_1").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_2",      F.col("proporcao_aluno_nivel_2").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_3",      F.col("proporcao_aluno_nivel_3").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_4",      F.col("proporcao_aluno_nivel_4").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_5",      F.col("proporcao_aluno_nivel_5").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_6",      F.col("proporcao_aluno_nivel_6").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_7",      F.col("proporcao_aluno_nivel_7").cast(DoubleType()))
    .withColumn("proporcao_aluno_nivel_8",      F.col("proporcao_aluno_nivel_8").cast(DoubleType()))
    .withColumn("_prata_data_processamento",    F.col("_prata_data_processamento").cast(TimestampType()))
    .withColumn("_versao_pipeline",             F.col("_versao_pipeline").cast(StringType()))
)
#
uf_silver.show(10)

Colunas antes: 18
Colunas depois: 15
Linhas antes: 145
Linhas depois: 145
Colunas antes: 15
Colunas depois: 17
+----+--------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------------+----------------+
| ano|sigla_uf|serie|rede|taxa_alfabetizacao|media_portugues|proporcao_aluno_nivel_0|proporcao_aluno_nivel_1|proporcao_aluno_nivel_2|proporcao_aluno_nivel_3|proporcao_aluno_nivel_4|proporcao_aluno_nivel_5|proporcao_aluno_nivel_6|proporcao_aluno_nivel_7|proporcao_aluno_nivel_8|_prata_data_processamento|_versao_pipeline|
+----+--------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+------------------

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/uf"
#
uf_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)

## Dados em streaming (simulados)

In [0]:
#
streaming_silver = leitura_streaming_bronze("destino_valido")
#
streaming_silver = remover_coluna_metadados_bronze(streaming_silver)
#
streaming_silver = remover_duplicados(streaming_silver)
#
streaming_silver = metadados_silver(streaming_silver)
#
streaming_silver.show(10)

Colunas antes: 11
Colunas depois: 8
Linhas antes: 2736
Linhas depois: 2736
Colunas antes: 8
Colunas depois: 10
+--------------------+----+------------+--------+----+-----+------------------+---------------+-------------------------+----------------+
|                  id| ano|id_municipio|sigla_uf|rede|serie|taxa_alfabetizacao|media_portugues|_prata_data_processamento|_versao_pipeline|
+--------------------+----+------------+--------+----+-----+------------------+---------------+-------------------------+----------------+
|e8e0aaa2-33e4-43a...|2025|     3509502|      SP|   4|    2|              62.3|          524.7|     2026-09-22 01:00:...|      v1.0_spark|
|239e007a-2c6b-409...|2025|     3509502|      SP|   3|    2|              49.8|          644.6|     2026-09-22 01:00:...|      v1.0_spark|
|25320d9d-1e6e-47f...|2025|     3509502|      SP|   3|    2|              64.3|          918.6|     2026-09-22 01:00:...|      v1.0_spark|
|3e2bea56-ed80-4eb...|2025|     3550308|      SP|   3| 

In [0]:
#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/streaming"
#
streaming_silver.write.partitionBy("ano").mode("overwrite").parquet(file_path)